# SolanaSentinel — Backtest de Umbrales ML

**Pregunta central:**  
> *"¿Qué habría pasado si hubiera exigido un ML score mínimo diferente al sniper?"*

## Qué analiza este notebook

- Usa **solo los registros de `sniper_positions`** (posiciones reales de simulación)
- Recorre todos los umbrales de ML score de 0 a 100 y recalcula el P&L
- Muestra win rate, ROI, drawdown, Sharpe y nº de trades por umbral
- Identifica el umbral óptimo según distintos criterios
- Analiza falsos negativos: tokens que el ML rechazó y que sí subieron

**Nota:** El análisis es sobre posiciones ya simuladas (las que pasaron anti-scam, liquidez, etc.)  
Bajar el umbral ML no añade posiciones nuevas — solo filtra dentro de lo que el sniper ya procesó.

## Nota: modelo v2 activo desde 2026-04-27

El modelo v2 usa `outcome_max_gain_pct >= 20` como label (en vez de ratio_1h).
Threshold recomendado: **67** (min_pump_score en sniper config).
PR-AUC: 0.152 vs 0.108 anterior (+40%).
Las posiciones anteriores usan la escala del modelo v1 — comparar con cautela.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

plt.style.use('dark_background')
ACCENT  = '#00d4ff'
GREEN   = '#00ff88'
RED     = '#ff4444'
PURPLE  = '#a78bfa'
YELLOW  = '#fbbf24'
ORANGE  = '#fb923c'
PINK    = '#f472b6'
GRAY    = '#6b7280'

DB_PATH = Path('../backend/data/sentinel.db')
assert DB_PATH.exists(), f'DB no encontrada: {DB_PATH}'
Path('charts').mkdir(exist_ok=True)
print(f'DB: {DB_PATH}')

## 1. Carga de datos

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Posiciones del sniper con datos de outcome de detected_tokens
df_pos = pd.read_sql_query("""
    SELECT
        sp.id, sp.token_mint, sp.token_symbol, sp.platform,
        sp.entry_price, sp.entry_time, sp.entry_mc,
        sp.simulated_sol, sp.simulated_sol_net,
        sp.fee_entry_sol, sp.fee_entry_pct, sp.fee_exit_pct,
        sp.current_price, sp.pnl_percent, sp.pnl_sol, sp.fees_sol,
        sp.status, sp.exit_price, sp.exit_time, sp.exit_reason,
        sp.ml_score,
        dt.pump_probability,
        dt.outcome_price_1h, dt.outcome_price_6h, dt.outcome_price_24h,
        dt.outcome_max_price, dt.outcome_max_gain_pct, dt.outcome_complete,
        dt.initial_liquidity, dt.market_cap, dt.risk_score
    FROM sniper_positions sp
    LEFT JOIN detected_tokens dt ON sp.token_mint = dt.token_mint
    ORDER BY sp.entry_time
""", conn)

# Tokens rechazados por ML (para analizar falsos negativos)
df_fn = pd.read_sql_query("""
    SELECT
        token_mint, token_symbol, platform, detected_at,
        ml_score, pump_probability,
        initial_liquidity, market_cap, risk_score,
        COALESCE(NULLIF(price_usd, 0), latest_price_usd) AS price_usd,
        outcome_price_1h, outcome_price_6h, outcome_price_24h,
        outcome_max_gain_pct, outcome_complete,
        reject_reason
    FROM detected_tokens
    WHERE reject_reason LIKE '%ml_score%'
       OR reject_reason LIKE '%pump_score%'
       OR reject_reason LIKE '%ml%'
    ORDER BY detected_at
""", conn)

conn.close()

# ─── Limpieza y columnas derivadas ───────────────────────────────────────────
df_pos['entry_time'] = pd.to_datetime(df_pos['entry_time'], errors='coerce')
df_pos['exit_time']  = pd.to_datetime(df_pos['exit_time'],  errors='coerce')
df_pos['ml_score']   = pd.to_numeric(df_pos['ml_score'],    errors='coerce')

# Duración de la posición en minutos
df_pos['duration_min'] = (
    (df_pos['exit_time'] - df_pos['entry_time']).dt.total_seconds() / 60
).clip(lower=0)

# Ratio 1h usando datos de outcome (para comparar con TP/SL simulado)
eps = 1e-12
df_pos['ratio_1h'] = df_pos['outcome_price_1h'] / (df_pos['entry_price'] + eps)
df_pos['pnl_1h_pct'] = (df_pos['ratio_1h'] - 1) * 100

# P&L efectivo: usa simulación para cerradas, pnl_percent para abiertas
df_pos['pnl_eff'] = df_pos['pnl_percent'].fillna(0)

has_ml   = df_pos['ml_score'].notna()
n_closed = (df_pos['status'] != 'open').sum()
n_open   = (df_pos['status'] == 'open').sum()

print('=' * 56)
print('  RESUMEN DE DATOS')
print('=' * 56)
print(f'  Total posiciones:   {len(df_pos):,}')
print(f'    cerradas:         {n_closed:,}  ({n_closed/max(len(df_pos),1)*100:.0f}%)')
print(f'    abiertas:         {n_open:,}  ({n_open/max(len(df_pos),1)*100:.0f}%)')
print(f'  Con ml_score:       {has_ml.sum():,}  ({has_ml.mean()*100:.0f}%)')
print(f'  Rechazadas por ML:  {len(df_fn):,}')
if has_ml.sum() > 0:
    print(f'\n  ML score: min={df_pos.ml_score.min():.0f}  '
          f'median={df_pos.ml_score.median():.0f}  '
          f'max={df_pos.ml_score.max():.0f}')
print(f'\n  Status breakdown:')
print(df_pos.status.value_counts().to_string())

## 2. Rendimiento actual (sin filtro adicional de ML)

In [ ]:
def compute_stats(positions: pd.DataFrame, label: str = '') -> dict:
    """Calcula estadísticas de trading para un conjunto de posiciones."""
    if len(positions) == 0:
        return {'n': 0, 'label': label}

    pnl   = positions['pnl_eff'].dropna()
    sol   = positions['pnl_sol'].dropna()
    wins  = (pnl > 0).sum()
    total = len(pnl)

    # Sharpe simplificado (sin tasa libre de riesgo)
    sharpe = (pnl.mean() / pnl.std()) if pnl.std() > 0 else 0

    # Max drawdown sobre curva de equity acumulada
    cumulative = (1 + pnl / 100).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = ((cumulative - rolling_max) / rolling_max * 100)
    max_dd = drawdown.min()

    # Expectancy: avg win * win_rate + avg_loss * loss_rate
    avg_win  = pnl[pnl > 0].mean() if (pnl > 0).any() else 0
    avg_loss = pnl[pnl < 0].mean() if (pnl < 0).any() else 0
    win_rate = wins / total if total > 0 else 0
    expectancy = avg_win * win_rate + avg_loss * (1 - win_rate)

    return {
        'label':       label,
        'n':           total,
        'win_rate':    win_rate,
        'avg_pnl':     pnl.mean(),
        'median_pnl':  pnl.median(),
        'total_sol':   sol.sum(),
        'avg_sol':     sol.mean(),
        'best':        pnl.max(),
        'worst':       pnl.min(),
        'sharpe':      sharpe,
        'max_dd':      max_dd,
        'expectancy':  expectancy,
        'avg_win':     avg_win,
        'avg_loss':    avg_loss,
    }

base_stats = compute_stats(df_pos, 'Todos (sin filtro ML)')

print('Rendimiento con TODAS las posiciones (ml_score >= 0):')
print(f'  Trades:       {base_stats["n"]:,}')
print(f'  Win rate:     {base_stats["win_rate"]*100:.1f}%')
print(f'  Avg P&L:      {base_stats["avg_pnl"]:+.2f}%')
print(f'  Median P&L:   {base_stats["median_pnl"]:+.2f}%')
print(f'  Total SOL:    {base_stats["total_sol"]:+.4f} SOL')
print(f'  Best trade:   {base_stats["best"]:+.1f}%')
print(f'  Worst trade:  {base_stats["worst"]:+.1f}%')
print(f'  Sharpe:       {base_stats["sharpe"]:.3f}')
print(f'  Max Drawdown: {base_stats["max_dd"]:+.1f}%')
print(f'  Expectancy:   {base_stats["expectancy"]:+.2f}% por trade')

## 3. Distribución de ML scores

In [ ]:
df_ml = df_pos[df_pos['ml_score'].notna()].copy()

if len(df_ml) == 0:
    print('⚠ No hay posiciones con ml_score registrado.')
    print('  Posibles causas:')
    print('  - El ML estaba desactivado cuando se generaron las posiciones')
    print('  - ml_score = NULL en todas las filas (revisar sniper config)')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Distribución de ML scores en posiciones del sniper', fontsize=14, y=1.02)

    # Histograma de scores
    ax = axes[0]
    bins = np.arange(0, 105, 5)
    wins_hist  = df_ml[df_ml['pnl_eff'] > 0]['ml_score']
    loss_hist  = df_ml[df_ml['pnl_eff'] <= 0]['ml_score']
    ax.hist(loss_hist, bins=bins, alpha=0.7, color=RED,   label=f'Pérdida (n={len(loss_hist)})')
    ax.hist(wins_hist, bins=bins, alpha=0.7, color=GREEN, label=f'Ganancia (n={len(wins_hist)})')
    ax.set_title('Distribución de ML score\n(verde=ganó, rojo=perdió)')
    ax.set_xlabel('ML Score'); ax.set_ylabel('Nº posiciones')
    ax.legend(); ax.grid(alpha=0.15)

    # CDF del score
    ax = axes[1]
    sorted_scores = np.sort(df_ml['ml_score'].values)
    cdf = np.arange(1, len(sorted_scores) + 1) / len(sorted_scores)
    ax.plot(sorted_scores, (1 - cdf) * 100, color=ACCENT, lw=2)
    for th in [30, 50, 70]:
        pct = ((df_ml['ml_score'] >= th).sum() / len(df_ml)) * 100
        ax.axvline(th, color=YELLOW, ls='--', lw=0.8, alpha=0.6)
        ax.text(th + 1, pct + 2, f'{th}: {pct:.0f}% trades', fontsize=8, color=YELLOW)
    ax.set_title('% de trades retenidos\npor umbral de ML score')
    ax.set_xlabel('Umbral ML score'); ax.set_ylabel('% trades que pasan')
    ax.grid(alpha=0.15)

    # Boxplot por rango de score
    ax = axes[2]
    df_ml['score_bin'] = pd.cut(df_ml['ml_score'], bins=[0,20,40,60,80,100],
                                 labels=['0-20','20-40','40-60','60-80','80-100'])
    groups = [df_ml[df_ml['score_bin'] == b]['pnl_eff'].dropna().values
              for b in ['0-20','20-40','40-60','60-80','80-100']]
    bp = ax.boxplot(groups, labels=['0-20','20-40','40-60','60-80','80-100'],
                    patch_artist=True, medianprops={'color': YELLOW, 'lw': 2})
    colors_bp = [RED, ORANGE, YELLOW, ACCENT, GREEN]
    for patch, c in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.axhline(0, color='white', ls='--', lw=0.8)
    ax.set_title('P&L % por rango de score')
    ax.set_xlabel('Rango ML score'); ax.set_ylabel('P&L %')
    ax.grid(alpha=0.15)

    plt.tight_layout()
    plt.savefig('charts/50_score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Posiciones con ml_score: {len(df_ml):,}')

## 4. Scatter: ML score vs P&L real

In [ ]:
if len(df_ml) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Scatter: ml_score vs pnl_percent
    ax = axes[0]
    colors_s = [GREEN if p > 0 else RED for p in df_ml['pnl_eff']]
    ax.scatter(df_ml['ml_score'], df_ml['pnl_eff'],
               c=colors_s, alpha=0.6, s=30, edgecolors='none')
    ax.axhline(0, color='white', ls='--', lw=1)

    # Línea de tendencia
    if len(df_ml) > 5:
        z = np.polyfit(df_ml['ml_score'].fillna(0), df_ml['pnl_eff'].fillna(0), 1)
        p = np.poly1d(z)
        xs = np.linspace(df_ml['ml_score'].min(), df_ml['ml_score'].max(), 100)
        ax.plot(xs, p(xs), color=YELLOW, lw=2, ls='--', label=f'Tendencia (slope={z[0]:+.3f})')
        ax.legend()

    ax.set_title('ML Score vs P&L por trade\n(verde=ganó, rojo=perdió)')
    ax.set_xlabel('ML Score'); ax.set_ylabel('P&L %')
    ax.grid(alpha=0.15)

    # Win rate y avg P&L por decil de score
    ax = axes[1]
    df_ml['decil'] = pd.qcut(df_ml['ml_score'], q=min(10, len(df_ml)//5),
                              labels=False, duplicates='drop')
    decil_g = df_ml.groupby('decil').agg(
        win_rate  = ('pnl_eff', lambda x: (x > 0).mean()),
        avg_pnl   = ('pnl_eff', 'mean'),
        score_min = ('ml_score', 'min'),
        score_max = ('ml_score', 'max'),
        n         = ('pnl_eff', 'count'),
    ).reset_index()
    decil_g['range'] = decil_g.apply(lambda r: f'{r.score_min:.0f}-{r.score_max:.0f}', axis=1)

    x = np.arange(len(decil_g))
    w = 0.35
    bars1 = ax.bar(x - w/2, decil_g['win_rate']*100, w,
                   color=GREEN, alpha=0.8, label='Win rate %')
    ax2 = ax.twinx()
    bars2 = ax2.bar(x + w/2, decil_g['avg_pnl'], w,
                    color=[GREEN if v > 0 else RED for v in decil_g['avg_pnl']],
                    alpha=0.8, label='Avg P&L %')
    ax.axhline(50, color=YELLOW, ls=':', lw=1, label='50% win rate')
    ax.set_xticks(x); ax.set_xticklabels(decil_g['range'], rotation=45, ha='right', fontsize=8)
    ax.set_title('Win rate y Avg P&L por decil de score')
    ax.set_ylabel('Win rate (%)', color=GREEN)
    ax2.set_ylabel('Avg P&L (%)', color=ACCENT)
    ax2.axhline(0, color='white', ls='--', lw=0.8)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + [mpatches.Patch(color=ACCENT, label='Avg P&L %')],
              labels1 + ['Avg P&L %'], fontsize=8, loc='upper left')
    ax.grid(alpha=0.15)

    # Mostrar N en cada barra
    for i, row in decil_g.iterrows():
        ax.text(i - w/2, row['win_rate']*100 + 1, f'n={int(row.n)}',
                ha='center', fontsize=7, color='white')

    plt.tight_layout()
    plt.savefig('charts/51_score_vs_pnl.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Barrido de umbrales — el análisis principal

Para cada umbral de ML score (0 a 100) calculamos:
- **Win rate**: % trades ganadores
- **Avg P&L**: media del P&L por trade
- **Total SOL**: suma del P&L en SOL
- **Sharpe**: retorno/riesgo
- **Nº trades**: posiciones que pasarían el filtro

In [ ]:
# Barrido completo de umbrales
thresholds  = np.arange(0, 101, 1)
sweep_rows  = []

for th in thresholds:
    if df_ml.empty:
        subset = df_pos  # sin filtro si no hay ml_score
    else:
        subset = df_ml[df_ml['ml_score'] >= th]

    if len(subset) == 0:
        sweep_rows.append({
            'threshold': th, 'n': 0, 'win_rate': np.nan,
            'avg_pnl': np.nan, 'total_sol': np.nan,
            'sharpe': np.nan, 'max_dd': np.nan,
            'expectancy': np.nan,
        })
        continue

    pnl  = subset['pnl_eff'].dropna()
    sol  = subset['pnl_sol'].dropna()
    n    = len(pnl)
    wins = (pnl > 0).sum()

    sharpe = (pnl.mean() / pnl.std()) if (n > 1 and pnl.std() > 0) else 0
    cum    = (1 + pnl / 100).cumprod()
    roll   = cum.cummax()
    max_dd = ((cum - roll) / roll * 100).min() if n > 0 else 0

    avg_win  = pnl[pnl > 0].mean() if (pnl > 0).any() else 0
    avg_loss = pnl[pnl < 0].mean() if (pnl < 0).any() else 0
    wr       = wins / n

    sweep_rows.append({
        'threshold':  th,
        'n':          n,
        'win_rate':   wr * 100,
        'avg_pnl':    pnl.mean(),
        'total_sol':  sol.sum(),
        'sharpe':     sharpe,
        'max_dd':     max_dd,
        'expectancy': avg_win * wr + avg_loss * (1 - wr),
    })

sweep = pd.DataFrame(sweep_rows)
sweep_valid = sweep[sweep['n'] >= 3]  # mínimo 3 trades para estadística estable

print(f'Barrido completado: {len(sweep)} umbrales evaluados')
print(f'Con ≥3 trades: {len(sweep_valid)} umbrales')
if not sweep_valid.empty:
    best_wr    = sweep_valid.loc[sweep_valid['win_rate'].idxmax()]
    best_pnl   = sweep_valid.loc[sweep_valid['avg_pnl'].idxmax()]
    best_sh    = sweep_valid.loc[sweep_valid['sharpe'].idxmax()]
    best_exp   = sweep_valid.loc[sweep_valid['expectancy'].idxmax()]
    print(f'\n  Mejor win rate:    umbral={best_wr.threshold:.0f}  '
          f'win={best_wr.win_rate:.1f}%  n={best_wr.n:.0f}')
    print(f'  Mejor avg P&L:     umbral={best_pnl.threshold:.0f}  '
          f'avg={best_pnl.avg_pnl:+.2f}%  n={best_pnl.n:.0f}')
    print(f'  Mejor Sharpe:      umbral={best_sh.threshold:.0f}  '
          f'sharpe={best_sh.sharpe:.3f}  n={best_sh.n:.0f}')
    print(f'  Mejor expectancy:  umbral={best_exp.threshold:.0f}  '
          f'exp={best_exp.expectancy:+.2f}%  n={best_exp.n:.0f}')

In [ ]:
# ─── Visualización principal del barrido ────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle('Backtest de umbrales ML — ¿Qué habría pasado con ml_score >= X?',
             fontsize=15, y=1.01)
gs = gridspec.GridSpec(3, 3, hspace=0.4, wspace=0.35)

th    = sweep['threshold']
valid = sweep['n'] >= 3

def mark_opt(ax, series, maximize=True):
    """Marca el umbral óptimo en un eje."""
    s = series[valid]
    if s.dropna().empty: return
    idx = s.idxmax() if maximize else s.idxmin()
    opt_th = sweep.loc[idx, 'threshold']
    opt_v  = series[idx]
    ax.axvline(opt_th, color=YELLOW, ls=':', lw=1.5, alpha=0.8)
    ax.annotate(f' ★{opt_th:.0f}', xy=(opt_th, opt_v), fontsize=8,
                color=YELLOW, xytext=(3, 3), textcoords='offset points')

# ── 1. Nº de trades ───────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ax.fill_between(th, sweep['n'], alpha=0.3, color=ACCENT)
ax.plot(th, sweep['n'], color=ACCENT, lw=2)
ax.set_title('Nº de trades'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('Trades'); ax.grid(alpha=0.15)
# Marcar los umbrales clave
for t in [30, 50, 70]:
    n_t = sweep.loc[sweep['threshold'] == t, 'n'].values
    if len(n_t): ax.axvline(t, color=GRAY, ls='--', lw=0.8)

# ── 2. Win rate ───────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
wr_plot = sweep['win_rate'].where(valid)
ax.plot(th, wr_plot, color=GREEN, lw=2)
ax.fill_between(th, wr_plot, 50, where=wr_plot >= 50, alpha=0.2, color=GREEN)
ax.fill_between(th, wr_plot, 50, where=wr_plot < 50,  alpha=0.2, color=RED)
ax.axhline(50, color='white', ls='--', lw=1, label='50%')
ax.set_title('Win rate (%)'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('Win rate %'); ax.legend(fontsize=8); ax.grid(alpha=0.15)
mark_opt(ax, sweep['win_rate'])

# ── 3. Avg P&L por trade ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
avg_plot = sweep['avg_pnl'].where(valid)
ax.plot(th, avg_plot, color=ACCENT, lw=2)
ax.fill_between(th, avg_plot, 0, where=avg_plot >= 0, alpha=0.2, color=GREEN)
ax.fill_between(th, avg_plot, 0, where=avg_plot < 0,  alpha=0.2, color=RED)
ax.axhline(0, color='white', ls='--', lw=1)
ax.set_title('Avg P&L por trade (%)'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('P&L %'); ax.grid(alpha=0.15)
mark_opt(ax, sweep['avg_pnl'])

# ── 4. Total P&L en SOL ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
sol_plot = sweep['total_sol'].where(valid)
ax.plot(th, sol_plot, color=PURPLE, lw=2)
ax.fill_between(th, sol_plot, 0, where=sol_plot >= 0, alpha=0.2, color=GREEN)
ax.fill_between(th, sol_plot, 0, where=sol_plot < 0,  alpha=0.2, color=RED)
ax.axhline(0, color='white', ls='--', lw=1)
ax.set_title('Total P&L (SOL)'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('SOL'); ax.grid(alpha=0.15)
mark_opt(ax, sweep['total_sol'])

# ── 5. Sharpe ratio ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
sh_plot = sweep['sharpe'].where(valid)
ax.plot(th, sh_plot, color=YELLOW, lw=2)
ax.axhline(0, color='white', ls='--', lw=1)
ax.axhline(1, color=GREEN, ls=':', lw=1, label='Sharpe=1 (bueno)')
ax.set_title('Sharpe ratio'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('Sharpe'); ax.legend(fontsize=8); ax.grid(alpha=0.15)
mark_opt(ax, sweep['sharpe'])

# ── 6. Max Drawdown ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
dd_plot = sweep['max_dd'].where(valid)
ax.plot(th, dd_plot, color=RED, lw=2)
ax.fill_between(th, dd_plot, 0, alpha=0.2, color=RED)
ax.set_title('Max Drawdown (%)'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('Drawdown %'); ax.grid(alpha=0.15)
mark_opt(ax, sweep['max_dd'], maximize=False)

# ── 7. Expectancy ────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
exp_plot = sweep['expectancy'].where(valid)
ax.plot(th, exp_plot, color=ORANGE, lw=2)
ax.fill_between(th, exp_plot, 0, where=exp_plot >= 0, alpha=0.2, color=GREEN)
ax.fill_between(th, exp_plot, 0, where=exp_plot < 0,  alpha=0.2, color=RED)
ax.axhline(0, color='white', ls='--', lw=1)
ax.set_title('Expectancy (% esperado por trade)'); ax.set_xlabel('Umbral ML score')
ax.set_ylabel('Expectancy %'); ax.grid(alpha=0.15)
mark_opt(ax, sweep['expectancy'])

# ── 8. Score compuesto (normalizado) ─────────────────────────────────────
ax = fig.add_subplot(gs[2, 1:])
# Score compuesto = 0.35*win_rate_norm + 0.35*avg_pnl_norm + 0.30*sharpe_norm
def norm01(s):
    mn, mx = s.min(), s.max()
    if mx == mn: return s * 0
    return (s - mn) / (mx - mn)

sv = sweep_valid.copy()
if len(sv) >= 3:
    sv['score_comp'] = (
        0.35 * norm01(sv['win_rate']) +
        0.35 * norm01(sv['avg_pnl']) +
        0.30 * norm01(sv['sharpe'])
    )
    best_comp = sv.loc[sv['score_comp'].idxmax()]
    bar_c = [GREEN if v == sv['score_comp'].max() else ACCENT for v in sv['score_comp']]
    ax.bar(sv['threshold'], sv['score_comp'], color=bar_c, alpha=0.8, width=1)
    ax.axvline(best_comp['threshold'], color=YELLOW, ls='--', lw=2,
               label=f'Óptimo compuesto = {best_comp["threshold"]:.0f}')
    ax.set_title('Puntuación compuesta (0.35×win_rate + 0.35×avg_pnl + 0.30×sharpe)\n'
                 'Verde = umbral óptimo global')
    ax.set_xlabel('Umbral ML score'); ax.set_ylabel('Score normalizado [0-1]')
    ax.legend(fontsize=10); ax.grid(alpha=0.15)
else:
    ax.text(0.5, 0.5, 'Datos insuficientes para score compuesto\n(necesita ≥3 trades por umbral)',
            transform=ax.transAxes, ha='center', va='center', fontsize=12)

plt.savefig('charts/52_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparativa detallada: umbrales clave

In [ ]:
# Umbrales a comparar: 0 (sin filtro), 30, 50, 70, 85 + óptimo compuesto
compare_thresholds = [0, 30, 50, 70, 85]
if len(sweep_valid) > 0:
    sv['score_comp'] = (
        0.35 * norm01(sv['win_rate'].fillna(0)) +
        0.35 * norm01(sv['avg_pnl'].fillna(0)) +
        0.30 * norm01(sv['sharpe'].fillna(0))
    ) if 'score_comp' not in sv.columns else sv['score_comp']
    opt_th = int(sv.loc[sv['score_comp'].idxmax(), 'threshold'])
    if opt_th not in compare_thresholds:
        compare_thresholds.append(opt_th)

compare_stats = []
for th in sorted(compare_thresholds):
    if df_ml.empty:
        subset = df_pos
    else:
        subset = df_ml[df_ml['ml_score'] >= th]
    if len(subset) == 0: continue
    s = compute_stats(subset, f'≥{th}')
    s['threshold'] = th
    compare_stats.append(s)

df_comp = pd.DataFrame(compare_stats)

# Tabla comparativa
print('═' * 88)
print(f'  {"Umbral":<10} {"Trades":>7} {"Win %":>7} {"Avg P&L":>9} {"Total SOL":>11}  '
      f'{"Sharpe":>7} {"MaxDD":>8} {"Expect":>8}')
print('─' * 88)
for _, r in df_comp.iterrows():
    flag = ' ★' if r.get('threshold') == (opt_th if 'opt_th' in dir() else -1) else '  '
    print(f'{flag}≥{int(r["threshold"]):<8} {r["n"]:>7,.0f} {r["win_rate"]*100:>6.1f}% '
          f'{r["avg_pnl"]:>+8.2f}%  {r["total_sol"]:>+9.4f} SOL  '
          f'{r["sharpe"]:>7.3f} {r["max_dd"]:>7.1f}% {r["expectancy"]:>+7.2f}%')
print('═' * 88)
print('★ = umbral óptimo compuesto')

# Gráfico de radar/barras comparativo
if len(df_comp) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Comparativa directa de umbrales ML', fontsize=13)
    c_palette = [GRAY, ORANGE, YELLOW, ACCENT, GREEN, PINK, PURPLE]
    labels_comp = [f'≥{int(r["threshold"])}' for _, r in df_comp.iterrows()]

    # Win rate
    ax = axes[0]
    bars = ax.bar(labels_comp, df_comp['win_rate']*100,
                  color=c_palette[:len(df_comp)], edgecolor='none', alpha=0.85)
    ax.axhline(50, color='white', ls='--', lw=1)
    for bar, v in zip(bars, df_comp['win_rate']*100):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%',
                ha='center', fontsize=9)
    ax.set_title('Win rate (%)'); ax.set_ylabel('%'); ax.grid(alpha=0.15)

    # Avg P&L
    ax = axes[1]
    bars = ax.bar(labels_comp, df_comp['avg_pnl'],
                  color=[GREEN if v > 0 else RED for v in df_comp['avg_pnl']],
                  edgecolor='none', alpha=0.85)
    ax.axhline(0, color='white', ls='--', lw=1)
    for bar, v in zip(bars, df_comp['avg_pnl']):
        ax.text(bar.get_x()+bar.get_width()/2,
                v + (0.2 if v >= 0 else -0.5), f'{v:+.2f}%',
                ha='center', fontsize=9)
    ax.set_title('Avg P&L por trade (%)'); ax.set_ylabel('%'); ax.grid(alpha=0.15)

    # Total SOL
    ax = axes[2]
    sol_vals = df_comp['total_sol'].fillna(0)
    bars = ax.bar(labels_comp, sol_vals,
                  color=[GREEN if v > 0 else RED for v in sol_vals],
                  edgecolor='none', alpha=0.85)
    ax.axhline(0, color='white', ls='--', lw=1)
    for bar, v in zip(bars, sol_vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                v + (0.0002 if v >= 0 else -0.0005), f'{v:+.4f}',
                ha='center', fontsize=9)
    ax.set_title('Total P&L (SOL)'); ax.set_ylabel('SOL'); ax.grid(alpha=0.15)

    plt.tight_layout()
    plt.savefig('charts/53_threshold_compare.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Curvas de equity acumulada por umbral

In [ ]:
show_thresholds = [0, 30, 50, 70]
if 'opt_th' in dir() and opt_th not in show_thresholds:
    show_thresholds.append(opt_th)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
c_lines = [GRAY, ORANGE, YELLOW, ACCENT, GREEN, PINK]

ax = axes[0]
for th, col in zip(sorted(show_thresholds), c_lines):
    if df_ml.empty:
        subset = df_pos.sort_values('entry_time')
    else:
        subset = df_ml[df_ml['ml_score'] >= th].sort_values('entry_time')
    if len(subset) < 2: continue
    pnl = subset['pnl_eff'].fillna(0)
    equity = (1 + pnl / 100).cumprod()
    ax.plot(range(len(equity)), equity, color=col, lw=2,
            label=f'≥{th}  (n={len(subset)})')

ax.axhline(1.0, color='white', ls='--', lw=0.8, label='Capital inicial')
ax.set_title('Curva de equity acumulada\n(base = 1.0 = capital inicial)')
ax.set_xlabel('Nº de trade'); ax.set_ylabel('Multiplicador de capital')
ax.legend(fontsize=9); ax.grid(alpha=0.15)

# Distribución de retornos overlay
ax = axes[1]
bins = np.linspace(-20, 30, 60)
for th, col in zip(sorted(show_thresholds), c_lines):
    if df_ml.empty:
        subset = df_pos
    else:
        subset = df_ml[df_ml['ml_score'] >= th]
    if len(subset) < 2: continue
    pnl = subset['pnl_eff'].dropna()
    ax.hist(pnl.clip(-20, 30), bins=bins, alpha=0.4, density=True,
            color=col, label=f'≥{th}')

ax.axvline(0, color='white', ls='--', lw=1)
ax.set_title('Distribución de retornos por umbral')
ax.set_xlabel('P&L %'); ax.set_ylabel('Densidad')
ax.legend(fontsize=9); ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('charts/54_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Análisis por plataforma y hora del día

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Análisis contextual de posiciones', fontsize=13)

# ── 1. P&L por plataforma ─────────────────────────────────────────────────
ax = axes[0, 0]
platforms = df_pos['platform'].fillna('unknown').unique()
plat_data = []
for p in platforms:
    sub = df_pos[df_pos['platform'].fillna('unknown') == p]
    if len(sub) < 2: continue
    plat_data.append({'platform': p, 'avg_pnl': sub['pnl_eff'].mean(),
                      'win_rate': (sub['pnl_eff'] > 0).mean() * 100, 'n': len(sub)})
if plat_data:
    df_plat = pd.DataFrame(plat_data)
    x = np.arange(len(df_plat))
    ax.bar(x, df_plat['avg_pnl'],
           color=[GREEN if v > 0 else RED for v in df_plat['avg_pnl']],
           edgecolor='none', alpha=0.85)
    ax.axhline(0, color='white', ls='--', lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r.platform}\n(n={r.n})" for _, r in df_plat.iterrows()])
    for i, v in enumerate(df_plat['avg_pnl']):
        ax.text(i, v + 0.1 if v >= 0 else v - 0.4, f'{v:+.2f}%', ha='center', fontsize=9)
ax.set_title('Avg P&L por plataforma'); ax.set_ylabel('P&L %'); ax.grid(alpha=0.15)

# ── 2. Win rate por hora del día ─────────────────────────────────────────
ax = axes[0, 1]
df_pos['hour'] = df_pos['entry_time'].dt.hour
hour_g = df_pos.groupby('hour').agg(
    win_rate=('pnl_eff', lambda x: (x > 0).mean() * 100),
    avg_pnl=('pnl_eff', 'mean'),
    n=('pnl_eff', 'count')
).reset_index()
bar_c_h = [GREEN if v >= 50 else RED for v in hour_g['win_rate']]
ax.bar(hour_g['hour'], hour_g['win_rate'], color=bar_c_h, edgecolor='none', alpha=0.85)
ax.axhline(50, color=YELLOW, ls='--', lw=1)
ax.set_title('Win rate por hora UTC')
ax.set_xlabel('Hora (UTC)'); ax.set_ylabel('Win rate %')
ax.set_xticks(range(0, 24, 2)); ax.grid(alpha=0.15)

# ── 3. P&L por MC de entrada ─────────────────────────────────────────────
ax = axes[1, 0]
df_mc = df_pos[df_pos['entry_mc'].notna() & (df_pos['entry_mc'] > 0)].copy()
if len(df_mc) > 10:
    df_mc['mc_bin'] = pd.qcut(df_mc['entry_mc'], q=5, labels=False, duplicates='drop')
    mc_g = df_mc.groupby('mc_bin').agg(
        avg_pnl=('pnl_eff', 'mean'),
        win_rate=('pnl_eff', lambda x: (x > 0).mean() * 100),
        mc_min=('entry_mc', 'min'), mc_max=('entry_mc', 'max'),
        n=('pnl_eff', 'count')
    ).reset_index()
    mc_g['label'] = mc_g.apply(
        lambda r: f'${r.mc_min/1e3:.0f}k-${r.mc_max/1e3:.0f}k\n(n={int(r.n)})', axis=1
    )
    ax.bar(range(len(mc_g)), mc_g['avg_pnl'],
           color=[GREEN if v > 0 else RED for v in mc_g['avg_pnl']],
           edgecolor='none', alpha=0.85)
    ax.axhline(0, color='white', ls='--', lw=1)
    ax.set_xticks(range(len(mc_g)))
    ax.set_xticklabels(mc_g['label'], fontsize=8)
    ax.set_title('Avg P&L por Market Cap de entrada')
    ax.set_ylabel('P&L %'); ax.grid(alpha=0.15)
else:
    ax.text(0.5, 0.5, 'Datos insuficientes', transform=ax.transAxes,
            ha='center', va='center')

# ── 4. Heatmap: ML score vs P&L bucket ───────────────────────────────────
ax = axes[1, 1]
if len(df_ml) >= 20:
    df_ml_hm = df_ml.copy()
    df_ml_hm['pnl_bin']   = pd.cut(df_ml_hm['pnl_eff'],
                                     bins=[-100,-20,-10,-5,0,5,10,20,100],
                                     labels=['<-20','-20:-10','-10:-5','-5:0','0:5','5:10','10:20','>20'])
    df_ml_hm['score_bin'] = pd.cut(df_ml_hm['ml_score'],
                                    bins=[0,20,40,60,80,100],
                                    labels=['0-20','20-40','40-60','60-80','80-100'])
    hm = df_ml_hm.pivot_table(index='score_bin', columns='pnl_bin',
                                values='id', aggfunc='count', fill_value=0)
    cmap = LinearSegmentedColormap.from_list('rg', ['#1a1a2e', '#ff4444', '#00ff88'])
    im = ax.imshow(hm.values, aspect='auto', cmap=cmap, interpolation='nearest')
    ax.set_xticks(range(len(hm.columns)))
    ax.set_yticks(range(len(hm.index)))
    ax.set_xticklabels(hm.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(hm.index, fontsize=8)
    for i in range(len(hm.index)):
        for j in range(len(hm.columns)):
            if hm.values[i, j] > 0:
                ax.text(j, i, str(int(hm.values[i, j])),
                        ha='center', va='center', fontsize=8, color='white')
    plt.colorbar(im, ax=ax, label='Nº trades')
    ax.set_title('Heatmap: ML score vs P&L')
    ax.set_xlabel('Resultado (P&L %)')
    ax.set_ylabel('Rango ML score')
else:
    ax.text(0.5, 0.5, 'Datos insuficientes para heatmap\n(necesita ≥20 posiciones con ml_score)',
            transform=ax.transAxes, ha='center', va='center', fontsize=11)
    ax.set_title('Heatmap: ML score vs P&L')

plt.tight_layout()
plt.savefig('charts/55_contextual.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Falsos negativos — tokens que el ML rechazó y subieron

In [ ]:
if len(df_fn) == 0:
    print('No hay tokens rechazados por ML con reject_reason registrado.')
    print('Posible causa: el filtro ML estaba desactivado (min_pump_score=0).')
else:
    eps = 1e-12
    df_fn['ratio_1h'] = df_fn['outcome_price_1h'] / (df_fn['price_usd'] + eps)
    df_fn['pnl_1h'] = (df_fn['ratio_1h'] - 1) * 100
    df_fn_complete = df_fn[df_fn['outcome_complete'] == 1].copy()

    print(f'Tokens rechazados por ML:  {len(df_fn):,}')
    print(f'Con outcome completo (1h): {len(df_fn_complete):,}')

    if len(df_fn_complete) > 0:
        pumped_fn = (df_fn_complete['ratio_1h'] >= 1.20).sum()
        print(f'\nDe los rechazados:')
        print(f'  Subieron +20% en 1h:  {pumped_fn} ({pumped_fn/len(df_fn_complete)*100:.1f}%)')
        print(f'  Avg P&L 1h:           {df_fn_complete["pnl_1h"].mean():+.1f}%')
        print(f'  Mejor:                {df_fn_complete["pnl_1h"].max():+.1f}%')

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('Análisis de falsos negativos (rechazados por ML)', fontsize=12)

        # Distribución de P&L de rechazados
        ax = axes[0]
        pnl_fn = df_fn_complete['pnl_1h'].clip(-50, 200)
        ax.hist(pnl_fn, bins=40, color=ORANGE, alpha=0.8, edgecolor='none')
        ax.axvline(0,   color='white', ls='--', lw=1, label='Sin cambio')
        ax.axvline(20,  color=GREEN,   ls=':',  lw=1, label='+20% (pump)')
        ax.set_title('Distribución de retorno 1h\nde tokens rechazados por ML')
        ax.set_xlabel('P&L % (1h)'); ax.set_ylabel('Nº tokens')
        ax.legend(); ax.grid(alpha=0.15)

        # ML score de rechazados vs su outcome
        ax = axes[1]
        df_fn_sc = df_fn_complete[df_fn_complete['ml_score'].notna()]
        if len(df_fn_sc) > 0:
            c_fn = [GREEN if p >= 20 else RED for p in df_fn_sc['pnl_1h']]
            ax.scatter(df_fn_sc['ml_score'], df_fn_sc['pnl_1h'],
                       c=c_fn, alpha=0.6, s=35, edgecolors='none')
            ax.axhline(20, color=GREEN, ls=':', lw=1, label='+20% pump')
            ax.axhline(0,  color='white', ls='--', lw=1)
            ax.set_title('ML score del rechazado vs retorno real 1h\n'
                         '(verde = hubiera sido un pump)')
            ax.set_xlabel('ML score asignado')
            ax.set_ylabel('P&L real 1h (%)')
            ax.legend(); ax.grid(alpha=0.15)
        else:
            ax.text(0.5, 0.5, 'Sin ml_score en rechazados',
                    transform=ax.transAxes, ha='center', va='center')

        plt.tight_layout()
        plt.savefig('charts/56_false_negatives.png', dpi=150, bbox_inches='tight')
        plt.show()

## 10. Análisis del outcome real 1h vs TP/SL simulado

In [ ]:
# Comparar P&L de simulación (TP/SL) vs P&L real si hubieras aguantado 1h
df_out = df_pos[
    df_pos['outcome_price_1h'].notna() &
    df_pos['entry_price'].notna() &
    (df_pos['entry_price'] > 0)
].copy()

print(f'Posiciones con outcome 1h disponible: {len(df_out):,}')

if len(df_out) >= 5:
    df_out['pnl_1h_real'] = (df_out['outcome_price_1h'] / df_out['entry_price'] - 1) * 100

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Simulación con TP/SL vs aguantar 1 hora', fontsize=13)

    # Scatter simulación vs real
    ax = axes[0]
    ax.scatter(df_out['pnl_eff'], df_out['pnl_1h_real'],
               color=ACCENT, alpha=0.5, s=25, edgecolors='none')
    lim = max(abs(df_out['pnl_eff'].min()), abs(df_out['pnl_eff'].max()),
              abs(df_out['pnl_1h_real'].min()), abs(df_out['pnl_1h_real'].max())) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'w--', lw=0.8, label='y=x (igual resultado)')
    ax.axhline(0, color=GRAY, lw=0.5); ax.axvline(0, color=GRAY, lw=0.5)
    ax.set_title('P&L simulación vs P&L real 1h')
    ax.set_xlabel('P&L con TP/SL (%)')
    ax.set_ylabel('P&L real 1h (%)')
    ax.legend(fontsize=8); ax.grid(alpha=0.15)

    # Distribución comparada
    ax = axes[1]
    bins = np.linspace(-25, 30, 50)
    ax.hist(df_out['pnl_eff'].clip(-25, 30), bins=bins,
            alpha=0.6, color=ACCENT, label='TP/SL simulado')
    ax.hist(df_out['pnl_1h_real'].clip(-25, 30), bins=bins,
            alpha=0.6, color=GREEN, label='Hold 1h')
    ax.axvline(0, color='white', ls='--', lw=1)
    avg_sim = df_out['pnl_eff'].mean()
    avg_1h  = df_out['pnl_1h_real'].mean()
    ax.axvline(avg_sim, color=ACCENT, ls=':', lw=1.5, label=f'Avg sim={avg_sim:+.2f}%')
    ax.axvline(avg_1h,  color=GREEN,  ls=':', lw=1.5, label=f'Avg 1h={avg_1h:+.2f}%')
    ax.set_title('Distribución: TP/SL vs Hold 1h')
    ax.set_xlabel('P&L %'); ax.set_ylabel('Nº posiciones')
    ax.legend(fontsize=8); ax.grid(alpha=0.15)

    # Mejor umbral ML por P&L real 1h (sin TP/SL)
    ax = axes[2]
    if df_ml.empty:
        df_out_ml = df_out
    else:
        df_out_ml = df_out[df_out['ml_score'].notna()]

    if len(df_out_ml) > 0:
        thresholds_1h = np.arange(0, 91, 5)
        avg_1h_by_th  = []
        for t in thresholds_1h:
            sub = df_out_ml if df_out_ml.equals(df_out) else df_out_ml[df_out_ml['ml_score'] >= t]
            if len(sub) < 3:
                avg_1h_by_th.append(np.nan)
            else:
                avg_1h_by_th.append(sub['pnl_1h_real'].mean())
        ax.plot(thresholds_1h, avg_1h_by_th, color=GREEN, lw=2,
                label='Hold 1h')
        ax.axhline(0, color='white', ls='--', lw=1)
        ax.set_title('Avg P&L real 1h (sin TP/SL)\npor umbral ML')
        ax.set_xlabel('Umbral ML score')
        ax.set_ylabel('Avg P&L 1h %')
        ax.legend(fontsize=8); ax.grid(alpha=0.15)
    else:
        ax.text(0.5, 0.5, 'Sin datos de ml_score + outcome',
                transform=ax.transAxes, ha='center', va='center')

    plt.tight_layout()
    plt.savefig('charts/57_sim_vs_1h.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nResumen comparativo:')
    print(f'  Avg P&L TP/SL:   {df_out["pnl_eff"].mean():+.2f}%')
    print(f'  Avg P&L hold 1h: {df_out["pnl_1h_real"].mean():+.2f}%')
    print(f'  La estrategia TP/SL es {"mejor" if avg_sim > avg_1h else "peor"} que hold 1h')

## 11. Recomendación final y tabla resumen

In [ ]:
print('=' * 70)
print('  RECOMENDACIÓN DE UMBRAL ML')
print('=' * 70)

if len(sweep_valid) == 0:
    print('  ⚠ Datos insuficientes para hacer una recomendación.')
    print('  Necesitas posiciones con ml_score registrado y cerradas (TP/SL).')
else:
    sv_clean = sweep_valid.dropna(subset=['win_rate','avg_pnl','sharpe'])
    if len(sv_clean) == 0:
        print('  ⚠ No hay umbrales con suficientes datos.')
    else:
        sv_clean = sv_clean.copy()
        sv_clean['score_comp'] = (
            0.35 * norm01(sv_clean['win_rate']) +
            0.35 * norm01(sv_clean['avg_pnl']) +
            0.30 * norm01(sv_clean['sharpe'])
        )
        best = sv_clean.loc[sv_clean['score_comp'].idxmax()]
        best_wr = sv_clean.loc[sv_clean['win_rate'].idxmax()]
        best_ap = sv_clean.loc[sv_clean['avg_pnl'].idxmax()]
        best_sh = sv_clean.loc[sv_clean['sharpe'].idxmax()]

        print(f'\n  Umbral óptimo compuesto:  {int(best.threshold)}')
        print(f'    Win rate:      {best.win_rate:.1f}%')
        print(f'    Avg P&L:       {best.avg_pnl:+.2f}%')
        print(f'    Total SOL:     {best.total_sol:+.4f}')
        print(f'    Sharpe:        {best.sharpe:.3f}')
        print(f'    Trades:        {int(best.n)}')

        print(f'\n  Si solo priorizas win rate: umbral = {int(best_wr.threshold)}')
        print(f'  Si priorizas avg P&L:       umbral = {int(best_ap.threshold)}')
        print(f'  Si priorizas Sharpe:        umbral = {int(best_sh.threshold)}')

        print(f'\n  Para activar en el sniper:')
        print(f'    → Ir a Sniper → Configuración → ML Pump Filter')
        print(f'    → Cambiar "Min Pump Score" a {int(best.threshold)}')
        print()
        print('  ⚠ Nota: con pocas posiciones los resultados no son estadísticamente')
        print('    significativos. Cuantas más posiciones tengas, más fiable.')

print()
print('  Tablas de posiciones por umbral (resumen de 10 en 10):')
print(f'  {"Umbral":<10} {"Trades":>7} {"Win %":>7} {"Avg PnL":>9} {"Total SOL":>11}  {"Sharpe":>7}')
print('  ' + '-' * 55)
for _, r in sweep[sweep['threshold'] % 10 == 0].iterrows():
    if pd.isna(r['win_rate']): continue
    print(f'  ≥{int(r.threshold):<9} {r.n:>7.0f} {r.win_rate:>6.1f}% '
          f'{r.avg_pnl:>+8.2f}%  {r.total_sol:>+9.4f} SOL  {r.sharpe:>7.3f}')